# Detection EDA: how large are the objects, and which training scale range covers them

Reads any prepared dataset in the detection format (e.g. the output of `data_pipeline.coco_to_detection`) and measures
the size of every face and plate box in pixels of the image as the model receives it.

The model sees fixed tiles of `MODEL_INPUT_PX` pixels made of `TOKEN_PX`-pixel tokens. Training resizes each image by a
random factor from `scale_range` (config `[training]`), so the sizes the model learns are the measured sizes times
those factors. Two limits matter:

| limit | meaning |
|---|---|
| smaller than one token (`TOKEN_PX`) | the object fits inside a single token: hard to find, and its outline is coarse |
| larger than one tile (`MODEL_INPUT_PX`) | no crop holds the whole object: the model only ever sees parts of it |

**Read off:** the size distribution per class (sections 2–3), the factors each percentile allows (section 4), and how a
candidate range spreads the sizes over the two limits (section 5). Put the chosen range into `scale_range`.

## 1. Settings

In [ ]:
DATASET_PATH = "data/v0_0_1/geico_detection_train"   # any dataset in the detection format
MODEL_INPUT_PX = 512        # grid x patch size: 32 x 16
TOKEN_PX = 16               # patch size of the backbone
SCALE_RANGE = (0.5, 2.0)    # candidate for scale_range in the config
PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
N_EXAMPLES = 8              # smallest and largest objects shown in section 6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk

from data_pipeline.formats import DETECTION_CLASSES
from trainer.common import resolve

ds = load_from_disk(str(resolve(DATASET_PATH)))
meta = ds.remove_columns("image").to_pandas()          # everything but the image bytes: fast
rows = []
for r in meta.itertuples():
    for (x, y, w, h), c in zip(r.objects["bbox"], r.objects["category"]):
        rows.append(dict(row=r.Index, image_id=r.image_id, subset=r.subset, cls=DETECTION_CLASSES[c],
                         x=x, y=y, w=w, h=h, W=r.width, H=r.height))
boxes = pd.DataFrame(rows, columns=["row", "image_id", "subset", "cls", "x", "y", "w", "h", "W", "H"])
boxes["min_side"] = boxes[["w", "h"]].min(axis=1)
boxes["max_side"] = boxes[["w", "h"]].max(axis=1)
boxes["share_of_image"] = boxes.max_side / boxes[["W", "H"]].max(axis=1)   # object's long side / image's long side
print(f"{len(meta)} images, {len(boxes)} boxes from {DATASET_PATH}")

## 2. Overview: images, boxes, image sizes

In [ ]:
n_obj = boxes.groupby("row").size()
display(pd.Series({
    "images": len(meta),
    "images with at least one box": int((n_obj > 0).sum()),
    "images without boxes": int(len(meta) - (n_obj > 0).sum()),
    "subsets": meta.subset.nunique(),
    **{f"boxes: {c}": int(n) for c, n in boxes.cls.value_counts().items()},
}).to_frame("count"))
display((meta.width.astype(str) + "x" + meta.height.astype(str)).value_counts().head(10).to_frame("images per size (WxH)"))

## 3. Object size in pixels

`min_side` decides whether an object is too small (under one token); `max_side` decides whether it is too large (over
one tile). `share_of_image` is the object's long side as a share of the image's long side: it is what stays fixed when
a whole image is resized at inference.

In [ ]:
def percentiles(col):
    return boxes.groupby("cls")[col].quantile(np.array(PERCENTILES) / 100).unstack().rename(columns=lambda q: f"p{q * 100:g}")

for col in ("min_side", "max_side", "share_of_image"):
    print(col); display(percentiles(col).round(3 if col == "share_of_image" else 1))

bins = np.logspace(0, np.log10(max(boxes.max_side.max(), MODEL_INPUT_PX) * 1.2), 50)
fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, ("min_side", "max_side")):
    for cls, g in boxes.groupby("cls"):
        ax.hist(g[col].clip(lower=1), bins=bins, alpha=0.6, label=f"{cls} ({len(g)})")
    ax.axvline(TOKEN_PX, color="k", ls="--", lw=1); ax.axvline(MODEL_INPUT_PX, color="k", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_xlabel(f"{col} (px)"); ax.set_title(f"{col}   (dashed: one token, dotted: one tile)")
axes[0].set_ylabel("boxes"); axes[0].legend(); plt.tight_layout(); plt.show()

## 4. The factors each percentile allows

For an object of a given size, a factor below `TOKEN_PX / min_side` shrinks it under one token, and a factor above
`MODEL_INPUT_PX / max_side` makes it larger than one tile. Per percentile:

- **lowest factor** = `TOKEN_PX / min_side`: the smallest factor that keeps an object of this size at least one token
- **highest factor** = `MODEL_INPUT_PX / max_side`: the largest factor that keeps it within one tile

A range is safe for a percentile where its low end is at least that percentile's lowest factor and its high end at most
its highest factor. Small objects push the low end up, large objects push the high end down.

In [ ]:
lo = (TOKEN_PX / percentiles("min_side")).round(2)
hi = (MODEL_INPUT_PX / percentiles("max_side")).round(2)
print("lowest factor (object stays >= one token)"); display(lo)
print("highest factor (object stays <= one tile)"); display(hi)

## 5. Check a candidate range

Each box is drawn many times with a random factor from `SCALE_RANGE`, evenly on a log scale as training draws it. The
table gives the share of training views of each class that fall under one token or over one tile; the plot shows the
sizes the model will learn next to the measured ones.

In [ ]:
rng = np.random.default_rng(0)
draws = 64
f = np.exp(rng.uniform(*np.log(SCALE_RANGE), size=(len(boxes), draws)))
seen_min, seen_max = boxes.min_side.values[:, None] * f, boxes.max_side.values[:, None] * f
cls = np.repeat(boxes.cls.values, draws)
check = pd.DataFrame({"under one token": (seen_min < TOKEN_PX).ravel(), "over one tile": (seen_max > MODEL_INPUT_PX).ravel(),
                      "cls": cls}).groupby("cls").mean().mul(100).round(1).add_suffix(" (%)")
print(f"SCALE_RANGE = {SCALE_RANGE}: model-pixel sizes the model trains on"); display(check)

bins = np.logspace(0, np.log10(max(seen_max.max(), MODEL_INPUT_PX) * 1.2), 50)
fig, axes = plt.subplots(1, boxes.cls.nunique(), figsize=(8 * boxes.cls.nunique(), 4), squeeze=False)
for ax, (c, g) in zip(axes[0], boxes.groupby("cls")):
    sel = np.repeat((boxes.cls == c).values, draws)
    ax.hist(g.max_side.clip(lower=1), bins=bins, density=True, alpha=0.5, label="measured")
    ax.hist(seen_max.ravel()[sel].clip(min=1), bins=bins, density=True, alpha=0.5, label=f"trained on, factors {SCALE_RANGE}")
    ax.axvline(TOKEN_PX, color="k", ls="--", lw=1); ax.axvline(MODEL_INPUT_PX, color="k", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("max_side (px)"); ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

## 6. Examples: the smallest and the largest objects

In [ ]:
import cv2
from datasets import Image as HFImage

raw = ds.cast_column("image", HFImage(decode=False))

def show(sel, title):
    if len(sel) == 0:
        return
    fig, axes = plt.subplots(1, len(sel), figsize=(2.2 * len(sel), 2.6), squeeze=False)
    for ax, b in zip(axes[0], sel.itertuples()):
        img = cv2.imdecode(np.frombuffer(raw[int(b.row)]["image"]["bytes"], np.uint8), cv2.IMREAD_COLOR)   # orientation applied
        r = int(max(b.w, b.h) * 0.75 + 8)
        cx, cy = int(b.x + b.w / 2), int(b.y + b.h / 2)
        crop = img[max(0, cy - r):cy + r, max(0, cx - r):cx + r]
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), interpolation="nearest")
        ax.set_title(f"{b.cls} {b.w:.0f}x{b.h:.0f}", fontsize=8); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

for c, g in boxes.groupby("cls"):
    show(g.nsmallest(N_EXAMPLES, "min_side"), f"{c}: smallest")
    show(g.nlargest(N_EXAMPLES, "max_side"), f"{c}: largest")